# When each chirp was received

Detects the pinger chirps directly in the WAV files of the three receivers and shows when each one arrived over
runs 1–2. Nothing is loaded from `out/`.

Method: the pipeline's FFT matched filter with the LFM replica (`detect.coarse`: 30→40 kHz, 100 ms, no Doppler
correction, channels summed on the USBL, threshold 20 dB above the running noise floor), then ping indexing by
unwrapping the ~1 s arrival phase (`associate.index_pings`), which also drops isolated false alarms.
Arrival times are to about a sample; the sub-sample TOA, Doppler and multipath fit in `detect.py` are not needed here.

Times are buoy time (s since the buoy files start). The USBL is shifted by the config's coarse clock guess
(`clock_offset_guess` = 102.7 s); `associate.py` refines it to 102.82 s from the data.

Plots are plotly: drag to zoom, double-click to reset, click legend entries to hide traces.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "processing"))
import config, recordings
from detect import coarse, MatchedFilter
from associate import index_pings

cfg = config.load()
names = [r.name for r in cfg.receivers]
recs = {n: recordings.load(cfg.receiver(n)) for n in names}
runs = [(r.name, r.start, r.end) for r in cfg.runs]
t0, t1 = 0.0, max(r.end for r in cfg.runs if r.pinger)  # the pinger is on before run 1 starts
xlim = (0, max(r[2] for r in runs))
colors = dict(zip(names, ["#1f77b4", "#ff7f0e", "#2ca02c"]))
for n, r in recs.items():
    print(f"{n:6} fs {r.fs} Hz, {r.n_ch} ch, {r.duration:.0f} s, clock offset {r.offset} s")

In [ ]:
def shade_runs(fig, rows="all"):
    for i, (name, a, b) in enumerate(runs):
        fig.add_vrect(x0=a, x1=b, fillcolor="grey", opacity=0.08 if i % 2 else 0.0, line_width=0,
                      row=rows, col="all")
    for name, a, b in runs:
        fig.add_annotation(x=(a + b) / 2, y=1.0, yref="paper", text=name, showarrow=False,
                           yanchor="bottom", font=dict(size=11, color="grey"))


def layout(fig, height, xtitle="buoy time [s]", **kw):
    fig.update_layout(template="plotly_white", height=height, margin=dict(l=70, r=20, t=40, b=50), **kw)
    xaxes = sorted((k for k in fig.layout if k.startswith("xaxis")), key=lambda k: int(k[5:] or 1))
    fig.layout[xaxes[-1]].title.text = xtitle          # bottom row of a stacked figure
    return fig

## A raw look
Spectrogram of a few seconds of the first buoy: the chirps are the 30→40 kHz up-sweeps, one per second.

In [ ]:
rec = recs[names[0]]
x, tx0 = rec.segment(200, 204, ch=0)
f, ts, Sxx = signal.spectrogram(x, fs=rec.fs, nperseg=1024, noverlap=768)
band = (f > 20e3) & (f < 48e3)
fig = go.Figure(go.Heatmap(x=tx0 + ts, y=f[band], z=(10 * np.log10(Sxx[band] + 1e-20)).astype(np.float32), colorscale="Magma",
                           zmin=-160, colorbar=dict(title="dB")))
layout(fig, 320, yaxis_title="frequency [Hz]").show()

## Matched-filter detection
Runs over runs 1–2 on every receiver (~45 s in total, mostly the 4-channel USBL).

In [ ]:
det = {}
for n in names:
    rec = recs[n]
    tic = time.time()
    a = coarse(rec, t0, t1, cfg.pinger, list(range(rec.n_ch)), thr_db=20.0)
    t = np.array([ai[0] for ai in a]) - rec.offset           # rx clock -> buoy time
    snr = np.array([ai[1] for ai in a])
    k, keep = index_pings(t, cfg.pinger.interval)
    det[n] = dict(t=t, snr=snr, k=k, keep=keep)
    print(f"{n:6} {len(t)} peaks, {keep.sum()} kept as pings, {np.median(snr[keep]):.1f} dB median SNR, "
          f"{time.time() - tic:.0f} s")

## A common ping index
The ping period is fitted on the first buoy (arrival time vs. its own ping index). Every arrival then gets a common
index `round((t − t_ref)/Tp)`; this works because travel-time differences between receivers stay well under half a period.

In [ ]:
d0 = det[names[0]]
Tp, _ = np.polyfit(d0["k"][d0["keep"]], d0["t"][d0["keep"]], 1)
all_t = np.concatenate([d["t"][d["keep"]] for d in det.values()])
t_ref = d0["t"][d0["keep"]][0]
frac = (all_t - t_ref) / Tp
t_ref += Tp * np.median(frac - np.round(frac))             # centre the schedule on the arrivals
for d in det.values():
    d["kc"] = np.round((d["t"] - t_ref) / Tp).astype(int)
n_ping = max(d["kc"][d["keep"]].max() for d in det.values()) + 1
t_ping = t_ref + np.arange(n_ping) * Tp
got = np.zeros((n_ping, len(names)), bool)
for i, n in enumerate(names):
    d = det[n]
    got[d["kc"][d["keep"]], i] = True
print(f"Tp = {Tp:.6f} s, {n_ping} ping slots from {t_ping[0]:.1f} to {t_ping[-1]:.1f} s")

## Matched-filter output
`mf_amplitude` runs the same FFT matched filter as the detector over any stretch of a recording. The replica is
normalized so a chirp of amplitude A gives |c| = A at its start, so the output is in the WAV's units (buoys:
full scale = 1). On the USBL the amplitude is the rms over the 4 channels. `peak_hold` reduces it to the maximum per
block, so long stretches stay light enough to plot without hiding any chirp.

In [ ]:
def mf_amplitude(rec, ta, tb, chunk=8.0):
    """|matched-filter output| between buoy times ta..tb (rms over channels): (t, amp)."""
    p, fs = cfg.pinger, rec.fs
    mf = MatchedFilter(p, fs, int(chunk * fs) + int(1.1 * p.duration * fs))
    ts, amps = [], []
    for c0 in np.arange(ta, tb, chunk):
        c1 = min(c0 + chunk, tb)
        x, tx0 = rec.segment(c0, c1 + 1.05 * p.duration)
        n = min(int(round((c1 - c0) * fs)), len(x) - mf.nrep)
        if n <= 0:
            continue
        c = mf.out(mf.xspec(x), 1.0, n)
        amps.append(np.sqrt(np.mean(np.abs(c) ** 2, axis=1)))
        ts.append(tx0 + np.arange(n) / fs)
    return np.concatenate(ts), np.concatenate(amps)


def peak_hold(t, a, nb):
    """Maximum of a per block of nb samples, at the time of that maximum."""
    m = len(a) // nb
    A = a[:m * nb].reshape(m, nb)
    i = A.argmax(1) + np.arange(m) * nb
    return t[i], a[i]


def db(a):
    return 20 * np.log10(a)

### A window at full rate
Set `ta, tb` to any window. Each trace is capped at `max_pts` points by peak-hold, so windows up to ~0.4 s are at
full rate (every sample) and longer ones keep every peak but not the shape between samples. Markers are the detected
arrivals: each chirp collapses to a sharp peak at its start. Zoom in to see the echoes behind each peak.

In [ ]:
ta, tb = 200.0, 204.0
max_pts = 40_000
fig = make_subplots(rows=len(names), cols=1, shared_xaxes=True, vertical_spacing=0.04)
for r, n in enumerate(names, 1):
    t, a = mf_amplitude(recs[n], ta, tb)
    tp, ap = peak_hold(t, a, max(len(a) // max_pts, 1))
    fig.add_trace(go.Scattergl(x=tp, y=ap.astype(np.float32), mode="lines", line=dict(width=1, color=colors[n]), name=n), row=r, col=1)
    d = det[n]
    m = d["keep"] & (d["t"] > ta) & (d["t"] < tb)
    i = np.searchsorted(t, d["t"][m])
    fig.add_trace(go.Scatter(x=d["t"][m], y=a[np.clip(i, 0, len(a) - 1)], mode="markers",
                             marker=dict(symbol="x", size=8, color="black"), name=f"{n} detections",
                             showlegend=r == 1), row=r, col=1)
    fig.update_yaxes(title_text=f"{n} |c|", row=r, col=1)
layout(fig, 650, showlegend=False).show()

### One ping up close
−5 to +40 ms around a detected arrival, in dB relative to its peak. The first peak is the direct path; the later ones
are surface/bottom echoes (buoys) and the wall echo ~1 ms behind the USBL. The sidelobe width is ~1/B = 0.1 ms.

In [ ]:
t_ping0 = t_ping[np.argmin(np.abs(t_ping - 202.0))]
fig = make_subplots(rows=1, cols=len(names), shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=names)
for c, n in enumerate(names, 1):
    d = det[n]
    tk = d["t"][d["keep"]][np.argmin(np.abs(d["t"][d["keep"]] - t_ping0))]
    t, a = mf_amplitude(recs[n], tk - 0.05, tk + 0.06)
    m = (t > tk - 0.005) & (t < tk + 0.04)
    fig.add_trace(go.Scatter(x=1e3 * (t[m] - tk), y=db(a[m] / a[m].max()), mode="lines",
                             line=dict(width=1, color=colors[n]), name=f"{n} @ {tk:.4f} s"), row=1, col=c)
    fig.update_xaxes(title_text="time after arrival [ms]", row=1, col=c)
fig.update_yaxes(title_text="|c| [dB re peak]", range=[-40, 2], row=1, col=1)
fig.update_layout(template="plotly_white", height=350, margin=dict(l=70, r=20, t=40, b=50)).show()

### Amplitude vs time over runs 1–2
Peak-hold of |c| in 10 ms blocks (grey), the noise floor (median per 1 s, black), and the detected chirps (dots: the
peak within ±5 ms of each detection; hover for ping index and SNR). The chirp amplitude follows range and propagation.
For sample-level detail in a stretch, use the full-rate window above.

In [ ]:
env = {}
for n in names:
    tic = time.time()
    t, a = mf_amplitude(recs[n], t0, t1)
    fs = recs[n].fs
    tp, ap = peak_hold(t, a, int(0.01 * fs))
    n1 = fs                                          # 1 s blocks
    floor = np.median(a[:len(a) // n1 * n1].reshape(-1, n1), axis=1)
    d = det[n]
    m = d["keep"]
    i = np.round((d["t"][m] - t[0]) * fs).astype(int)
    w = int(0.005 * fs)
    pk = np.array([a[max(j - w, 0):j + w + 1].max() for j in i])
    env[n] = dict(t=tp, pk=ap, tf=t[:len(a) // n1 * n1:n1] + 0.5, floor=floor, td=d["t"][m], pkd=pk,
                  kd=d["kc"][m], snrd=d["snr"][m])
    print(f"{n:6} {time.time() - tic:.0f} s")

fig = make_subplots(rows=len(names), cols=1, shared_xaxes=True, vertical_spacing=0.04)
for r, n in enumerate(names, 1):
    e = env[n]
    first = r == 1
    fig.add_trace(go.Scattergl(x=e["t"].astype(np.float32), y=db(e["pk"]).astype(np.float32), mode="lines", line=dict(width=0.6, color="#aaaaaa"),
                               name="|c| peak-hold, 10 ms", legendgroup="pk", showlegend=first), row=r, col=1)
    fig.add_trace(go.Scatter(x=e["tf"], y=db(e["floor"]), mode="lines", line=dict(width=1.2, color="black"),
                             name="noise floor (1 s median)", legendgroup="fl", showlegend=first), row=r, col=1)
    fig.add_trace(go.Scattergl(x=e["td"], y=db(e["pkd"]), mode="markers", marker=dict(size=4, color=colors[n]),
                               name=f"{n} chirps", customdata=np.c_[e["kd"], e["snrd"]],
                               hovertemplate="t %{x:.3f} s<br>|c| %{y:.1f} dB<br>ping %{customdata[0]}"
                                             "<br>SNR %{customdata[1]:.1f} dB<extra></extra>"), row=r, col=1)
    fig.update_yaxes(title_text=f"{n} |c| [dB re FS]", row=r, col=1)
shade_runs(fig)
fig.update_xaxes(range=xlim)
layout(fig, 800, legend=dict(orientation="h", y=-0.08)).show()

## Detection timeline
One tick per detected chirp, coloured by matched-filter SNR. Red crosses are peaks rejected as false alarms.

In [ ]:
snr_all = np.concatenate([d["snr"][d["keep"]] for d in det.values()])
vmin, vmax = np.percentile(snr_all, [2, 98])
fig = go.Figure()
for i, n in enumerate(names):
    d = det[n]
    m = d["keep"]
    fig.add_trace(go.Scatter(x=d["t"][m], y=np.full(m.sum(), i), mode="markers", name=n, showlegend=False,
                             marker=dict(symbol="line-ns", size=22, line=dict(width=1.5, color=d["snr"][m]),
                                         color=d["snr"][m], cmin=vmin, cmax=vmax, colorscale="Viridis",
                                         showscale=i == 0, colorbar=dict(title="SNR [dB]")),
                             customdata=np.c_[d["kc"][m], d["snr"][m]],
                             hovertemplate=n + " t %{x:.3f} s<br>ping %{customdata[0]}<br>SNR %{customdata[1]:.1f} dB"
                                           "<extra></extra>"))
    fig.add_trace(go.Scatter(x=d["t"][~m], y=np.full((~m).sum(), i + 0.3), mode="markers", showlegend=False,
                             marker=dict(symbol="x", size=6, color="red"), name=f"{n} rejected"))
shade_runs(fig, rows=None)
fig.update_yaxes(tickvals=list(range(len(names))), ticktext=names, range=[-0.6, len(names) - 0.2])
fig.update_xaxes(range=xlim)
layout(fig, 320).show()

## Arrival time relative to the ping schedule
`t − (t_ref + k·Tp)`: the slow drift is the travel time changing as the boat moves; the offset between receivers
is the TDOA (for the USBL, plus the residual of its coarse clock offset).

In [ ]:
fig = go.Figure()
for n in names:
    d = det[n]
    m = d["keep"]
    fig.add_trace(go.Scatter(x=d["t"][m], y=1e3 * (d["t"][m] - t_ping[d["kc"][m]]), mode="markers",
                             marker=dict(size=3, color=colors[n]), name=n, customdata=d["kc"][m],
                             hovertemplate="t %{x:.3f} s<br>%{y:.2f} ms<br>ping %{customdata}<extra>" + n + "</extra>"))
shade_runs(fig, rows=None)
fig.update_xaxes(range=xlim)
layout(fig, 420, yaxis_title="arrival − schedule [ms]").show()

## Coverage per ping

In [ ]:
n_rx = got.sum(1)
fig = go.Figure(go.Scatter(x=t_ping, y=n_rx, mode="lines", line=dict(shape="hvh", width=1, color="black"),
                           customdata=np.arange(n_ping), hovertemplate="ping %{customdata}<br>%{y} receivers<extra></extra>"))
shade_runs(fig, rows=None)
fig.update_xaxes(range=xlim)
fig.update_yaxes(tickvals=list(range(len(names) + 1)))
layout(fig, 260, yaxis_title="receivers").show()

print(f"{'':6}" + "".join(f"{n:>10}" for n in names) + f"{'all 3':>10}")
for name, a, b in runs:
    sel = (t_ping >= a) & (t_ping < b)
    if not sel.any():
        print(f"{name:6}  no pings")
        continue
    row = [f"{got[sel, i].mean():9.0%}" for i in range(len(names))] + [f"{(n_rx[sel] == 3).mean():9.0%}"]
    print(f"{name:6}" + " ".join(row) + f"   ({sel.sum()} pings)")

## Gaps
Runs of more than 3 consecutive missed pings, per receiver.

In [ ]:
for i, n in enumerate(names):
    miss = ~got[:, i]
    edges = np.flatnonzero(np.diff(np.r_[0, miss.astype(int), 0]))
    gaps = [(t_ping[a], t_ping[b - 1], b - a) for a, b in zip(edges[::2], edges[1::2]) if b - a > 3]
    print(f"{n:6}", "  ".join(f"{a:.0f}–{b:.0f} s ({c})" for a, b, c in gaps) or "none")